In [31]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# =========================
# Configuration
# =========================
INPUT_CSV = "/itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/features.csv"
OUTPUT_DIR = "/itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_COL = "setting"
FEATURE_COL = "method"

METRIC_SPECS = [
    ("auroc",          "AUROC"),
    ("top_recall_150", "Recall@150"),
    ("bedroc_150",     "BEDROC@150"),
    ("bio_sim_150",    "BioSim@150"),
    ("bio_sim_300",    "BioSim@300"),
]

MODEL_NAME_MAP = {
    "2019_mf_add_pred":    "MF",
    "2019_nn_all_pred":    "DNN",
    "2019_rf_renamed_pred":"RF",
    "df_gnn_occsvm_pred":  "SVM",
    "gcn":                 "GCN",
    "graphsage":           "GraphSAGE",
}

FEATURE_NAME_MAP = {
    "ppi_emb_df":    "PPI-Diff",
    "ppi_emb_dw":    "PPI-DW",
    "ppi_emb_n2v":   "PPI-N2V",
    "literature_emb":"Literature",
    "seq_emb_esm":   "Seq-ESM2",
    "seq_emb_port":  "Seq-Prot",
}

FEATURE_ORDER = ["PPI-Diff", "PPI-DW", "PPI-N2V", "Literature", "Seq-ESM2", "Seq-Prot"]

MODEL_ORDER   = ["SVM", "RF", "DNN", "GraphSAGE", "GCN", "MF"]

# =========================
# Utilities
# =========================
def prettify_labels(df):
    df = df.copy()
    df["model_pretty"]   = df[MODEL_COL].map(MODEL_NAME_MAP).fillna(df[MODEL_COL])
    df["feature_pretty"] = df[FEATURE_COL].map(FEATURE_NAME_MAP).fillna(df[FEATURE_COL])
    return df

# =========================
# Load data
# =========================
df = pd.read_csv(INPUT_CSV)
df = prettify_labels(df)

df["model_pretty"]   = pd.Categorical(df["model_pretty"],   categories=MODEL_ORDER,   ordered=True)
df["feature_pretty"] = pd.Categorical(df["feature_pretty"], categories=FEATURE_ORDER, ordered=True)
df = df.sort_values(["model_pretty", "feature_pretty"]).reset_index(drop=True)

print("Loaded shape:", df.shape)
print(df.head())

# =========================
# Pre-normalise each pivot to [0, 1] for colour only
# =========================
pivots = {}
normalized_pivots = {}

for metric_col, metric_label in METRIC_SPECS:
    pivot = (
        df.pivot(index="model_pretty", columns="feature_pretty", values=metric_col)
          .reindex(index=MODEL_ORDER, columns=FEATURE_ORDER)
    )
    pivots[metric_col] = pivot

    vals = pivot.values.astype(float)
    mn, mx = np.nanmin(vals), np.nanmax(vals)
    norm_vals = (vals - mn) / (mx - mn) if mx > mn else np.zeros_like(vals)
    normalized_pivots[metric_col] = norm_vals

# =========================
# Layout: row 0 → first 3 metrics, row 1 → BioSim pair
# =========================
ncols = 3
nrows = 2

# fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 5.2 * nrows))
axes[1, 2].axis("off")

used_axes = [axes[0, 0], axes[0, 1], axes[0, 2],
             axes[1, 0], axes[1, 1]]

cmap = plt.get_cmap("Blues")
# cmap = plt.get_cmap("PuBu")
# cmap = plt.get_cmap("GnBu")



for idx, (metric_col, metric_label) in enumerate(METRIC_SPECS):
    ax = used_axes[idx]
    pivot      = pivots[metric_col]
    norm_vals  = normalized_pivots[metric_col]

    # draw heatmap using normalised [0,1] colours
    im = ax.imshow(norm_vals, aspect="auto", cmap=cmap, vmin=0, vmax=1)

    # force the axes box itself to be square
    ax.set_box_aspect(1)

    ax.set_title(metric_label, fontsize=13, fontweight="bold", pad=8)
    ax.set_xticks(np.arange(len(FEATURE_ORDER)))
    ax.set_xticklabels(FEATURE_ORDER, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(MODEL_ORDER)))
    ax.set_yticklabels(MODEL_ORDER)

    # annotate with RAW values; underline the max
    max_val = np.nanmax(pivot.values)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                is_max = np.isclose(val, max_val)
                # use normalised value to decide text color
                brightness = norm_vals[i, j]
                text_color = "white" if brightness > 0.55 else "#333333"

                ax.text(
                    j, i, f"{val:.3f}",
                    ha="center", va="center",
                    fontsize=8, color=text_color,
                    fontweight="bold" if is_max else "normal",
                )
                if is_max:
                    ax.annotate(
                        "",
                        xy=(j + 0.28, i + 0.18),
                        xytext=(j - 0.28, i + 0.18),
                        arrowprops=dict(arrowstyle="-", color=text_color, lw=1.5),
                    )

    # for i in range(pivot.shape[0]):
    #     for j in range(pivot.shape[1]):
    #         val = pivot.iloc[i, j]
    #         if pd.notna(val):
    #             is_max = np.isclose(val, max_val)
    #             ax.text(
    #                 j, i, f"{val:.3f}",
    #                 ha="center", va="center",
    #                 # fontsize=8, color="dimgrey",
    #                 # fontsize=8, color="black",
    #                 fontsize=8, color="#696969",
    #                 fontweight="bold" if is_max else "normal",
    #             )
    #             if is_max:
    #                 ax.annotate(
    #                     "",
    #                     xy=(j + 0.28, i + 0.18),
    #                     xytext=(j - 0.28, i + 0.18),
    #                     # arrowprops=dict(arrowstyle="-", color="dimgrey", lw=1.5),
    #                     # arrowprops=dict(arrowstyle="-", color="black", lw=1.5),
    #                     arrowprops=dict(arrowstyle="-", color="#696969", lw=1.5))

# =========================
# Single shared colorbar
# =========================
sm = ScalarMappable(cmap=cmap, norm=Normalize(vmin=0, vmax=1))
sm.set_array([])

# attach to the hidden axes slot so it sits neatly to the right of BioSim@300
# instead of stealing from axes[1, 2]
cbar_ax = fig.add_axes([0.73, 0.11, 0.01, 0.3])  # [left, bottom, width, height]
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Min–max normalised score (per metric)", fontsize=10)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["0 (min)", "0.25", "0.50", "0.75", "1 (max)"])

plt.tight_layout(rect=[0, 0, 1, 0.98])

heatmap_path = os.path.join(OUTPUT_DIR, "all_heatmaps.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved: {heatmap_path}")

Loaded shape: (36, 13)
              setting          method  auroc  top_recall_150  top_recall_300  \
0  df_gnn_occsvm_pred      ppi_emb_df  0.808           0.221           0.263   
1  df_gnn_occsvm_pred      ppi_emb_dw  0.821           0.174           0.308   
2  df_gnn_occsvm_pred     ppi_emb_n2v  0.806           0.188           0.298   
3  df_gnn_occsvm_pred  literature_emb  0.735           0.106           0.186   
4  df_gnn_occsvm_pred     seq_emb_esm  0.658           0.092           0.140   

   bedroc_150  bedroc_300  succ_150  succ_300  bio_sim_150  bio_sim_300  \
0       0.142       0.204     0.479     0.521        0.168        0.160   
1       0.139       0.199     0.354     0.562        0.173        0.163   
2       0.141       0.197     0.375     0.521        0.170        0.160   
3       0.085       0.126     0.312     0.438        0.153        0.167   
4       0.067       0.095     0.312     0.417        0.142        0.147   

  model_pretty feature_pretty  
0          SV

/tmp/ipykernel_335575/416514512.py:186: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.98])


Saved: /itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/all_heatmaps.png


In [ ]:
# =========================
# Radar charts - normalized averaging only
# Apply to all six features for each model
#
# Output:
#   1) one big plot for Normalized Averaging
#
# In the big plot:
#   - 6 subplots total
#   - each subplot corresponds to one feature
#   - within each subplot, all models are shown
# =========================

radar_metrics = [
    "auroc",
    "top_recall_150",
    "top_recall_300",
    "bedroc_150",
    "bedroc_300",
    "succ_150",
    "succ_300",
    "bio_sim_150",
    "bio_sim_300",
]

# display names to match previous plots
RADAR_METRIC_LABELS = [
    "AUROC",
    "Recall@150",
    "Recall@300",
    "BEDROC@150",
    "BEDROC@300",
    "Succ@150",
    "Succ@300",
    "BioSim@150",
    "BioSim@300",
]


def normalize_columns_global(df_in, cols):
    """Min-max normalize each metric across all rows."""
    df_out = df_in.copy()
    for c in cols:
        cmin = df_out[c].min()
        cmax = df_out[c].max()
        if np.isclose(cmax, cmin):
            df_out[c] = 0.5
        else:
            df_out[c] = (df_out[c] - cmin) / (cmax - cmin)
    return df_out


def plot_big_radar_figure(df_plot, metric_cols, metric_labels, feature_order, model_order, title, output_path):
    """
    Create one big figure with 6 radar subplots.
    Each subplot = one feature
    Each line in subplot = one model
    """
    n_features = len(feature_order)
    ncols = 3
    nrows = 2

    labels = metric_labels
    num_vars = len(labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(18, 12),
        subplot_kw=dict(polar=True)
    )
    axes = np.array(axes).reshape(-1)

    for idx, feature in enumerate(feature_order):
        ax = axes[idx]
        sub = df_plot[df_plot["feature_pretty"] == feature].copy()

        sub["model_pretty"] = pd.Categorical(
            sub["model_pretty"],
            categories=model_order,
            ordered=True
        )
        sub = sub.sort_values("model_pretty")

        for _, row in sub.iterrows():
            values = row[metric_cols].tolist()
            values += values[:1]
            ax.plot(angles, values, linewidth=2, label=row["model_pretty"])
            ax.fill(angles, values, alpha=0.06)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_yticklabels([])
        ax.set_title(feature, pad=18, fontsize=12)

    # hide unused axes if any
    for idx in range(n_features, len(axes)):
        axes[idx].axis("off")

    # legend at bottom of whole figure
    handles, labels_legend = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_legend,
        title="Model",
        loc="lower center",
        ncol=len(model_order),
        bbox_to_anchor=(0.5, 0.02)
    )

    # fig.suptitle(title, fontsize=18, y=0.98)
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {output_path}")


# =========================
# Base radar input
# =========================
radar_base = df[["model_pretty", "feature_pretty"] + radar_metrics].copy()

radar_base["model_pretty"] = pd.Categorical(
    radar_base["model_pretty"],
    categories=MODEL_ORDER,
    ordered=True
)
radar_base["feature_pretty"] = pd.Categorical(
    radar_base["feature_pretty"],
    categories=FEATURE_ORDER,
    ordered=True
)
radar_base = radar_base.sort_values(["feature_pretty", "model_pretty"]).reset_index(drop=True)


# =====================================================
# Normalized Averaging only
# - normalize each metric globally
# - use normalized values directly for plotting
# - score = mean of normalized metrics
# =====================================================
radar_norm_avg = normalize_columns_global(radar_base, radar_metrics)
radar_norm_avg["selection_score"] = radar_norm_avg[radar_metrics].mean(axis=1)

# save score table
norm_avg_score_path = os.path.join(OUTPUT_DIR, "radar_normalized_averaging_scores.csv")
radar_norm_avg[["model_pretty", "feature_pretty", "selection_score"] + radar_metrics].to_csv(
    norm_avg_score_path, index=False
)
print(f"Saved: {norm_avg_score_path}")

plot_big_radar_figure(
    df_plot=radar_norm_avg,
    metric_cols=radar_metrics,
    metric_labels=RADAR_METRIC_LABELS,
    feature_order=FEATURE_ORDER,
    model_order=MODEL_ORDER,
    title="Radar Plots by Feature Across All Models (Normalized Averaging)",
    output_path=os.path.join(OUTPUT_DIR, "radar_bigplot_normalized_averaging.png")
)

print("Saved normalized averaging radar figure.")

Saved: /itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/radar_normalized_averaging_scores.csv
Saved: /itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/radar_bigplot_normalized_averaging.png
Saved normalized averaging radar figure.


In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("/itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/all_fusion.csv")

# Set index
df = df.set_index("Method")


# ---------------------------
# 2. Reorder columns
# ---------------------------
cols = [
    "AUROC",
    "Recall@150",
    "Recall@300",
    "BEDROC@150",
    "BEDROC@300",
    "Succ@150",
    "Succ@300",
    "BioSim@150",
    "BioSim@300"
]
df = df[cols]

# ---------------------------
# 3. Normalize per column
# ---------------------------
df_norm = (df - df.min()) / (df.max() - df.min())
df_norm = df_norm.fillna(0.0)

# ---------------------------
# 4. Split Kernel vs DNN
# ---------------------------
kernel_mask = df_norm.index.str.startswith("Kernel")
dnn_mask = df_norm.index.str.startswith("DNN")

df_kernel = df_norm[kernel_mask].copy()
df_dnn = df_norm[dnn_mask].copy()

df_kernel_raw = df[kernel_mask].copy()
df_dnn_raw = df[dnn_mask].copy()

# ---------------------------
# 5. Sort by fusion strategy + PPI grouping
# ---------------------------
fusion_order = ["Early", "Mid", "Late"]

def sort_by_fusion_and_ppi(df_part, df_raw_part):
    def fusion_rank(name):
        for i, key in enumerate(fusion_order):
            if key in name:
                return i
        return 99

    temp = pd.DataFrame(index=df_part.index)
    temp["fusion_rank"] = [fusion_rank(name) for name in df_part.index]
    temp["ppi_flag"] = df_part.index.str.contains("ppi", case=False)
    temp = temp.sort_values(["ppi_flag", "fusion_rank"])

    ordered_index = temp.index
    return df_part.loc[ordered_index], df_raw_part.loc[ordered_index]

df_kernel, df_kernel_raw = sort_by_fusion_and_ppi(df_kernel, df_kernel_raw)
df_dnn, df_dnn_raw = sort_by_fusion_and_ppi(df_dnn, df_dnn_raw)

# ---------------------------
# 6. Global best per column
# ---------------------------
best_methods = {col: df[col].idxmax() for col in df.columns}

# ---------------------------
# 7. Create equal-width subplots + shared colorbar
# ---------------------------
fig = plt.figure(figsize=(18, 8))
gs = fig.add_gridspec(
    1, 3,
    width_ratios=[1, 1, 0.05],
    wspace=0.5  # increased gap between Kernel and DNN panels
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

cmap = plt.cm.viridis
norm = Normalize(vmin=0, vmax=1)

# ---------------------------
# 8. Draw heatmaps manually
# ---------------------------
im1 = ax1.imshow(df_kernel.values, aspect="auto", cmap=cmap, norm=norm)
im2 = ax2.imshow(df_dnn.values, aspect="auto", cmap=cmap, norm=norm)

# Shared colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Normalized Score")

# ---------------------------
# 9. Axis formatting
# ---------------------------
for ax, df_part, title in [
    (ax1, df_kernel, "Kernel-based Models"),
    (ax2, df_dnn, "DNN-based Models"),
]:
    ax.set_title(title, fontsize=14)
    ax.set_xticks(range(len(df_part.columns)))
    ax.set_xticklabels(df_part.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(df_part.index)))
    ax.set_yticklabels(df_part.index)

    # Cell borders
    ax.set_xticks([x - 0.5 for x in range(1, len(df_part.columns))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(df_part.index))], minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)

ax1.set_ylabel("Methods")
ax1.set_xlabel("Metrics")
ax2.set_xlabel("Metrics")
ax2.set_ylabel("")

# push DNN y tick labels slightly away from its axis so they don't crowd center gap
ax2.tick_params(axis='y', pad=8)

# ---------------------------
# 10. Add PPI separators
# ---------------------------
split_k = sum(~df_kernel.index.str.contains("ppi", case=False))
split_d = sum(~df_dnn.index.str.contains("ppi", case=False))

if 0 < split_k < len(df_kernel):
    ax1.hlines(split_k - 0.5, -0.5, len(df_kernel.columns) - 0.5, colors="black", linewidth=2)

if 0 < split_d < len(df_dnn):
    ax2.hlines(split_d - 0.5, -0.5, len(df_dnn.columns) - 0.5, colors="black", linewidth=2)

# ---------------------------
# 11. Annotate exact values
#     Highest in each column = red
#     Others = black
# ---------------------------
def annotate_values(ax, df_raw_part):
    for i, method in enumerate(df_raw_part.index):
        for j, col in enumerate(df_raw_part.columns):
            value = df_raw_part.loc[method, col]
            text_color = "red" if method == best_methods[col] else "black"
            ax.text(
                j, i, f"{value:.3f}",
                ha="center", va="center",
                color=text_color, fontsize=9
            )

annotate_values(ax1, df_kernel_raw)
annotate_values(ax2, df_dnn_raw)

# ---------------------------
# 13. Final layout
# ---------------------------
# fig.suptitle(
#     "Normalized Performance Comparison: Kernel vs DNN Fusion Strategies",
#     fontsize=16
# )

plt.tight_layout(rect=[0.03, 0, 1, 0.96])
heatmap_path = os.path.join(OUTPUT_DIR, "all_fusion.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved: {heatmap_path}")

/tmp/ipykernel_3363626/1633676927.py:174: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.03, 0, 1, 0.96])


Saved: /itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/all_fusion.png
